# 🧠 Notebook 08: Strict Determinism Profile

## 1. Purpose + Scope

This notebook defines the "Strict Determinism Profile" (Tier A) which governs consensus-critical execution:

*   **Tier A Enforcement**: Rules for verifiable computation.
*   **Forbidden Operations**: What code cannot do in strict mode.
*   **Strict Float Division Trap**: Why float division might be disallowed.
*   **Symbol ID Trap**: Preventing leakage of local intern IDs.
*   **Time and Entropy Traps**: No access to wall-clock time or hardware RNG.

## 2. Spec References

*   `spec/determinism-profile.md`

## 3. Determinism Tier

**Tier A (Strict Determinism)**: The highest standard of reproducibility.

## 4. Reproducibility Setup

Ensure `t81_python` is built and available in `PYTHONPATH`.

In [ ]:
import sys
import os

build_dir = os.path.abspath(os.path.join(os.getcwd(), "../build"))
if build_dir not in sys.path:
    sys.path.append(build_dir)

try:
    import t81_python
    print("✅ t81_python loaded.")
except ImportError:
    print("❌ Failed to load t81_python.")
    sys.exit(1)

## 5. Forbidden Operations

In strict mode, operations like `random()`, `time()`, or iterating a map by insertion order are forbidden. If the VM supports setting a strict flag, these would trap.

In [ ]:
# Conceptual check of strict mode
def strict_mode_check(op_name):
    forbidden = ["random", "time", "map_insertion_order", "float_hardware_div"]
    if op_name in forbidden:
        raise RuntimeError(f"Trap: Operation '{op_name}' forbidden in Strict Mode (Tier A).")
    return "OK"

try:
    strict_mode_check("time")
except RuntimeError as e:
    print(f"✅ Caught expected trap: {e}")

## 6. Symbol ID Leakage

Code should never see the raw integer ID of a symbol, as it varies by process.

In [ ]:
# If we had a function `get_symbol_id("foo")`, it would be banned.
# Instead, code must use `hash("foo")` or work with the symbol opaque handle.
pass

## 7. Entropy Traps

Accessing hardware RNG is non-deterministic. Only deterministic PRNGs seeded from the transaction hash are allowed.

## 8. Architectural Commentary

Strict mode is what allows T81 to run on thousands of heterogeneous nodes and reach identical states. Without it, consensus is impossible.